# Modelo

**Objetivo:** treinar o LightGBM e comparar com os baselines na validação.

**Quatro versões:** com e sem colunas C (D11), com e sem agregações por cartão (D12). As colunas base estão na D15.

**Regras:**
- Treino só com os dados de treino e resultados só na validação. O teste não entra em nenhum cálculo.
- A decisão de quando parar o treino usa só os últimos 20% do treino (D14). A validação fica intacta.

In [1]:
from pathlib import Path
import re
import sys

import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
from divisao_temporal import calcular_cortes, dividir_por_tempo
from metricas import selecionar_revisadas, valor_capturado
from variaveis import agregacoes_anteriores, chave_de_cartao

ARQUIVO = RAIZ / "data" / "processed" / "treino.parquet"
nomes = pq.ParquetFile(ARQUIVO).schema_arrow.names
COLUNAS_C = [n for n in nomes if re.fullmatch(r"C\d+", n)]
COLUNAS_ID = [n for n in nomes if re.fullmatch(r"id_\d+", n)]
COLUNAS_CATEGORIA = ["ProductCD", "card4", "card6"]
COLUNAS_CARTAO = [f"card{i}" for i in range(1, 7)]
COLUNAS_TRANSACAO = ["card1", "card2", "card3", "card5", "addr1", "addr2", "dist1", "dist2"]
COLUNAS_EMAIL = ["P_emaildomain", "R_emaildomain"]

tabela = pd.read_parquet(
    ARQUIVO,
    columns=[
        "TransactionDT", "isFraud", "TransactionAmt",
        *dict.fromkeys([*COLUNAS_CATEGORIA, *COLUNAS_CARTAO, *COLUNAS_TRANSACAO]),
        *COLUNAS_EMAIL, *COLUNAS_C, *COLUNAS_ID,
    ],
)
tabela["tem_identidade"] = tabela[COLUNAS_ID].notna().any(axis=1).astype("int8")
tabela = tabela.drop(columns=COLUNAS_ID)

corte_treino, corte_validacao = calcular_cortes(tabela["TransactionDT"])
treino, validacao, _ = dividir_por_tempo(tabela, corte_treino, corte_validacao)
del _
tabela = pd.concat([treino, validacao])  # daqui em diante, sem o teste
del treino, validacao

# Agregações da D12: cada transação só enxerga instantes anteriores, sem isFraud.
chave = chave_de_cartao(tabela, [*COLUNAS_CARTAO, "addr1"])
agregadas = agregacoes_anteriores(tabela["TransactionDT"], tabela["TransactionAmt"], chave)
COLUNAS_AGREGADAS = list(agregadas.columns)
tabela = tabela.join(agregadas)
del agregadas, chave

no_treino = tabela["TransactionDT"] <= corte_treino
treino, validacao = tabela[no_treino], tabela[~no_treino]
del tabela
rotulos = validacao["isFraud"].astype("int8")
valores = validacao["TransactionAmt"].astype("float64")
print(f"Treino: {len(treino):,} | Validação: {len(validacao):,}")

Treino: 354,324 | Validação: 118,108


## 1. Baselines

Mesmo código do notebook 03, recalculado aqui para comparar tudo nos mesmos dados.

In [2]:
def preparar_logistica(parte: pd.DataFrame) -> pd.DataFrame:
    x = np.log1p(parte[COLUNAS_C].astype("float64"))
    x["log_valor"] = np.log1p(parte["TransactionAmt"].astype("float64"))
    x["tem_identidade"] = parte["tem_identidade"]
    for coluna in COLUNAS_CATEGORIA:
        x[coluna] = parte[coluna].astype("object").fillna("(vazio)")
    return x


def logistica(colunas_numericas: list[str]) -> pd.Series:
    modelo = Pipeline(
        [
            (
                "preparo",
                ColumnTransformer(
                    [
                        ("numericas", StandardScaler(), colunas_numericas),
                        ("categorias", OneHotEncoder(handle_unknown="ignore"), COLUNAS_CATEGORIA),
                    ]
                ),
            ),
            ("logistica", LogisticRegression(max_iter=1000)),
        ]
    )
    modelo.fit(preparar_logistica(treino), treino["isFraud"])
    return pd.Series(modelo.predict_proba(preparar_logistica(validacao))[:, 1], index=validacao.index)


pontuacoes = {
    "Regra por valor": valores,
    "Logística com C": logistica([*COLUNAS_C, "log_valor", "tem_identidade"]),
    "Logística sem C": logistica(["log_valor", "tem_identidade"]),
}

## 2. LightGBM

**Como o número de árvores é escolhido (D14):**
1. O treino é dividido no tempo: os primeiros 80% treinam, e os últimos 20% servem só para decidir quando parar.
2. O modelo adiciona árvores até a PR-AUC nesses 20% passar 100 árvores sem melhorar.
3. O modelo é treinado de novo com o treino inteiro, usando o número de árvores encontrado.

**Ajustes internos:** valores comuns e fixos, sem busca (D15). Os campos vazios ficam vazios, porque o LightGBM decide sozinho para que lado da árvore eles vão. Os textos entram como categorias.

In [3]:
PARAMETROS = {
    "objective": "binary",
    "metric": "average_precision",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "seed": 42,
    "deterministic": True,
    "force_row_wise": True,
    "num_threads": 4,
    "verbose": -1,
}
COLUNAS_BASE = ["TransactionAmt", "tem_identidade", *COLUNAS_CATEGORIA, *COLUNAS_TRANSACAO, *COLUNAS_EMAIL]

corte_parada, _ = calcular_cortes(treino["TransactionDT"], fracao_treino=0.8, fracao_validacao=0.2)
ajuste, parada, _ = dividir_por_tempo(treino, corte_parada, treino["TransactionDT"].max())
del _
print(f"Ajuste: {len(ajuste):,} | Parada: {len(parada):,}")

arvores = {}
for com_c in [False, True]:
    for com_agregadas in [False, True]:
        nome = f"LightGBM {'com' if com_c else 'sem'} C, {'com' if com_agregadas else 'sem'} agregações"
        colunas = [*COLUNAS_BASE, *(COLUNAS_C if com_c else []), *(COLUNAS_AGREGADAS if com_agregadas else [])]

        busca = lgb.train(
            PARAMETROS,
            lgb.Dataset(ajuste[colunas], ajuste["isFraud"]),
            num_boost_round=5000,
            valid_sets=[lgb.Dataset(parada[colunas], parada["isFraud"])],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
        arvores[nome] = busca.best_iteration

        final = lgb.train(PARAMETROS, lgb.Dataset(treino[colunas], treino["isFraud"]), num_boost_round=busca.best_iteration)
        pontuacoes[nome] = pd.Series(final.predict(validacao[colunas]), index=validacao.index)
        print(f"{nome}: {busca.best_iteration} árvores (PR-AUC na parada: {busca.best_score['valid_0']['average_precision']:.4f})")

Ajuste: 283,459 | Parada: 70,865


LightGBM sem C, sem agregações: 344 árvores (PR-AUC na parada: 0.2234)


LightGBM sem C, com agregações: 204 árvores (PR-AUC na parada: 0.2242)


LightGBM com C, sem agregações: 712 árvores (PR-AUC na parada: 0.5077)


LightGBM com C, com agregações: 501 árvores (PR-AUC na parada: 0.5003)


## 3. Qualidade geral na validação

PR-AUC: quanto maior, melhor. Brier: quanto menor, melhor. A regra por valor não produz probabilidade, então não tem Brier. Ordenando ao acaso, a PR-AUC fica perto da taxa de fraude da validação, 0,0390.

In [4]:
pd.DataFrame(
    {
        nome: {
            "PR-AUC": average_precision_score(rotulos, pontuacao),
            "Brier": np.nan if nome == "Regra por valor" else brier_score_loss(rotulos, pontuacao),
            "Árvores": arvores.get(nome, np.nan),
        }
        for nome, pontuacao in pontuacoes.items()
    }
).T.round(4)

,PR-AUC,Brier,Árvores
Regra por valor,0.0479,NaN,NaN
Logística com C,0.4357,0.0278,NaN
Logística sem C,0.1351,0.0356,NaN
"LightGBM sem C, sem agregações",0.2806,0.0322,344.0
"LightGBM sem C, com agregações",0.2718,0.0324,204.0
"LightGBM com C, sem agregações",0.5742,0.0232,712.0
"LightGBM com C, com agregações",0.5645,0.0235,501.0


## 4. Resultado por capacidade de revisão

Capacidade é a parte das compras da validação que a equipe consegue revisar. Todos os modelos ordenam pela chance de fraude. A ordenação que combina chance e valor fica para a Etapa 7.

In [5]:
linhas = []
for nome, pontuacao in pontuacoes.items():
    for capacidade in [0.005, 0.01, 0.03, 0.05]:
        revisadas = selecionar_revisadas(pontuacao, capacidade)
        linhas.append(
            {
                "Modelo": nome,
                "Capacidade (%)": 100 * capacidade,
                "Precisão (%)": 100 * rotulos.loc[revisadas].mean(),
                "Recall (%)": 100 * rotulos.loc[revisadas].sum() / rotulos.sum(),
                "Valor capturado (%)": valor_capturado(rotulos, pontuacao, valores, capacidade),
            }
        )
por_capacidade = pd.DataFrame(linhas).set_index(["Modelo", "Capacidade (%)"]).round(2)
por_capacidade

Precisão (%)  Recall (%)  \
Modelo                         Capacidade (%)                             
Regra por valor                0.5                     3.21        0.41   
                               1.0                     4.23        1.08   
                               3.0                     7.93        6.09   
                               5.0                     8.69       11.13   
Logística com C                0.5                    91.88       11.78   
                               1.0                    86.29       22.12   
                               3.0                    51.33       39.45   
                               5.0                    36.10       46.24   
Logística sem C                0.5                    26.57        3.40   
                               1.0                    25.04        6.42   
                               3.0                    20.17       15.51   
                               5.0                    17.80       22.79   
LightGBM sem C, sem agregações 0.5                    55.84        7.16   
                               1.0                    49.15       12.60   
                               3.0                    39.00       29.97   
                               5.0                    32.26       41.31   
LightGBM sem C, com agregações 0.5                    53.64        6.87   
                               1.0                    49.92       12.80   
                               3.0                    37.27       28.65   
                               5.0                    30.07       38.52   
LightGBM com C, sem agregações 0.5                    90.52       11.60   
                               1.0                    90.02       23.08   
                               3.0                    65.41       50.27   
                               5.0                    46.78       59.92   
LightGBM com C, com agregações 0.5                    91.37       11.71   
                               1.0                    89.93       23.05   
                               3.0                    65.27       50.16   
                               5.0                    46.22       59.21   

                                               Valor capturado (%)  
Modelo                         Capacidade (%)                       
Regra por valor                0.5                            6.73  
                               1.0                           12.14  
                               3.0                           37.60  
                               5.0                           51.83  
Logística com C                0.5                            3.84  
                               1.0                           11.39  
                               3.0                           26.65  
                               5.0                           34.48  
Logística sem C                0.5                            2.69  
                               1.0                            4.17  
                               3.0                            6.47  
                               5.0                            8.48  
LightGBM sem C, sem agregações 0.5                            7.38  
                               1.0                           12.83  
                               3.0                           23.81  
                               5.0                           33.15  
LightGBM sem C, com agregações 0.5                            8.04  
                               1.0                           13.43  
                               3.0                           24.57  
                               5.0                           31.11  
LightGBM com C, sem agregações 0.5                            6.26  
                               1.0                           14.39  
                               3.0                           40.26  
                               5.0                           53.52  
Li

## Conclusões do modelo

- O LightGBM supera a logística nas duas versões. Com colunas C, a PR-AUC sobe de 0,4357 para 0,5742. Sem colunas C, sobe de 0,1351 para 0,2806. Parte do ganho pode vir das colunas novas da D15, e não só do modelo.
- Com colunas C, o LightGBM também captura mais valor que a regra por valor em 3% e 5% das compras: 40,26% e 53,52%, contra 37,60% e 51,83%. Em 0,5%, a regra ainda captura mais: 6,73% contra 6,26%.
- Com colunas C, as agregações por cartão não mudaram o resultado de forma consistente. A PR-AUC foi de 0,5742 sem elas para 0,5645 com elas, e o valor capturado em 3% foi de 40,26% para 41,94%. Com uma única validação, essas diferenças não se distinguem de variação normal. Sem colunas C, o padrão se repete: 0,2806 e 0,2718.
- Como as agregações não trouxeram uma melhora clara, o modelo segue sem elas (D16). Isso evita guardar o histórico de cada cartão em produção e reduz o que precisa ser explicado.
- As colunas C continuam sendo o fator que mais pesa: sem elas, a PR-AUC do LightGBM cai de 0,5742 para 0,2806. Como não dá para auditar se usam informação futura, as duas versões seguem lado a lado (D11).
- **Limites:** os ajustes internos do LightGBM são fixos, sem busca. O número de árvores foi escolhido com 80% do treino (D14). Os custos de revisão ainda não entram. Estes resultados valem só para a validação, e o teste não foi usado.